## PS2 - campus route search

In [1]:
import heapq, time

pq = []
heapq.heappush(pq, (5, 'x'))
heapq.heappush(pq, (2, 'y'))
heapq.heappush(pq, (8, 'z'))
print(heapq.heappop(pq))
print(heapq.heappop(pq))  # pops smallest first, that's what f(n)=h(n) needs

(2, 'y')
(5, 'x')


In [2]:
def path_cost(graph, path):
    total = 0
    for i in range(len(path) - 1):
        for nb, c in graph[path[i]]:
            if nb == path[i + 1]:
                total += c
                break
    return total

### greedy best-first

In [3]:
def greedy_best_first(graph, h, start, goal):
    t0 = time.time()
    closed = set()
    parent = {}
    frontier = [(h[start], start)]
    nodes_expanded = 0
    while frontier:
        _, cur = heapq.heappop(frontier)
        if cur in closed:
            continue
        closed.add(cur)
        nodes_expanded += 1
        if cur == goal:
            path = [cur]
            while path[-1] != start:
                path.append(parent[path[-1]])
            path.reverse()
            return True, path, path_cost(graph, path), nodes_expanded, time.time() - t0
        for nb, cost in graph.get(cur, []):
            if nb not in closed:
                if nb not in parent:
                    parent[nb] = cur
                heapq.heappush(frontier, (h[nb], nb))
    return False, [], 0, nodes_expanded, time.time() - t0

In [4]:
toy_graph = {'X':[('Y',1),('Z',10)], 'Y':[('X',1),('Z',1)], 'Z':[('X',10),('Y',1)]}
toy_h = {'X':2, 'Y':1, 'Z':0}
print(greedy_best_first(toy_graph, toy_h, 'X', 'Z'))

(True, ['X', 'Z'], 10, 2, 0.0)


In [5]:
# went straight X -> Z above, h(goal) is always 0 so it wins instantly once Z is a direct neighbor.
# drop the direct edge so there's an actual choice between two hops
toy_graph = {'X':[('Y',1),('W',1)], 'Y':[('X',1),('Z',5)], 'W':[('X',1),('Z',5)], 'Z':[('Y',5),('W',5)]}
toy_h = {'X':2, 'Y':1, 'W':4, 'Z':0}
print(greedy_best_first(toy_graph, toy_h, 'X', 'Z'))  # X -> Y -> Z, Y's h beats W's

(True, ['X', 'Y', 'Z'], 6, 3, 0.0)


### a*

In [6]:
def a_star(graph, h, start, goal):
    t0 = time.time()
    best_g = {start: 0}
    parent = {}
    frontier = [(h[start], start)]
    closed = set()
    nodes_expanded = 0
    while frontier:
        f, cur = heapq.heappop(frontier)
        if cur in closed:
            continue
        closed.add(cur)
        nodes_expanded += 1
        if cur == goal:
            path = [cur]
            while path[-1] != start:
                path.append(parent[path[-1]])
            path.reverse()
            return True, path, best_g[goal], nodes_expanded, time.time() - t0
        for nb, cost in graph.get(cur, []):
            new_g = best_g[cur] + cost
            if nb not in best_g or new_g <= best_g[nb]:
                best_g[nb] = new_g
                parent[nb] = cur
                heapq.heappush(frontier, (new_g + h[nb], nb))
    return False, [], 0, nodes_expanded, time.time() - t0

### the real campus graph

In [7]:
graph = {
    'A':[('B',4),('C',2)], 'B':[('A',4),('D',5)],
    'C':[('A',2),('D',3),('E',6)], 'D':[('B',5),('C',3),('E',3)],
    'E':[('C',6),('D',3),('F',2)], 'F':[('E',2)]
}
h = {'A':7,'B':8,'C':5,'D':4,'E':2,'F':0}

In [8]:
print(greedy_best_first(graph, h, 'A', 'F'))

(True, ['A', 'C', 'E', 'F'], 10, 4, 0.0)


In [9]:
# goes A -> C -> E -> F, cost 10. assignment PDF's sample shows A -> C -> D -> E -> F for greedy
# but both routes cost 10 here (2+6+2 = 2+3+3+2), and E really does have the lower heuristic
print(a_star(graph, h, 'A', 'F'))  # matches the PDF exactly

(True, ['A', 'C', 'D', 'E', 'F'], 10, 5, 0.0)


### second graph, greedy actually loses this time

In [10]:
graph2 = {'S':[('A',1),('B',2)], 'A':[('S',1),('T',20)], 'B':[('S',2),('T',2)], 'T':[('A',20),('B',2)]}
h2 = {'S':3, 'A':1, 'B':2, 'T':0}
print('greedy:', greedy_best_first(graph2, h2, 'S', 'T'))
print('a star:', a_star(graph2, h2, 'S', 'T'))

greedy: (True, ['S', 'A', 'T'], 21, 3, 0.0)
a star: (True, ['S', 'B', 'T'], 4, 4, 0.0)


### full run

In [11]:
def print_result(name, found, path, cost, nodes, t):
    print(f'Algorithm: {name}')
    if not found:
        print('Path Found: No')
        return
    print('Path Found: Yes')
    print('Path: ' + ' -> '.join(path))
    print(f'Total Cost = {cost}')
    print(f'Nodes Expanded = {nodes}')
    print(f'Execution Time = {t:.6f}')

In [12]:
gf, gp, gc, gn, gt = greedy_best_first(graph, h, 'A', 'F')
print_result('Greedy Best-First Search', gf, gp, gc, gn, gt)
print()
af, ap, ac, an, at = a_star(graph, h, 'A', 'F')
print_result('A* Search', af, ap, ac, an, at)
print()
print('Comparison:')
print(f'Path cost -> Greedy = {gc}, A* = {ac}')
print(f'Nodes expanded -> Greedy = {gn}, A* = {an}')

Algorithm: Greedy Best-First Search
Path Found: Yes
Path: A -> C -> E -> F
Total Cost = 10
Nodes Expanded = 4
Execution Time = 0.000000

Algorithm: A* Search
Path Found: Yes
Path: A -> C -> D -> E -> F
Total Cost = 10
Nodes Expanded = 5
Execution Time = 0.000000

Comparison:
Path cost -> Greedy = 10, A* = 10
Nodes expanded -> Greedy = 4, A* = 5
